# 1. Definir las rutas del autoloader

In [0]:
%python

# Ruta de los archivos de entrada
source_path= "/Volumes/ecomotive/landing/sensores"

# Ubicación del esquema de Auto Loader
schema_path="/Volumes/ecomotive/metadata/schemas/sensores_pipeline/"

# Ubicación del checkpoint
checkpoint_path="/Volumes/ecomotive/metadata/checkpoints"

# Tabla destino donde se almacenarán los registros 
target_table = "ecomotive.bronze.iot_sensors_raw"


In [0]:
%python
print("Archivos disponibles:")

display(dbutils.fs.ls(source_path))

print("Ubicación del esquema:")
print(schema_path)

print("Ubicación del checkpoint:")
print(checkpoint_path)

Archivos disponibles:


path,name,size,modificationTime
dbfs:/Volumes/ecomotive/landing/sensores/iot_incremental_prueba_10_registros.csv,iot_incremental_prueba_10_registros.csv,958,1794190267000
dbfs:/Volumes/ecomotive/landing/sensores/iot_sensors_con_duplicados.csv,iot_sensors_con_duplicados.csv,1610,1794097802000
dbfs:/Volumes/ecomotive/landing/sensores/iot_sensors_extended.csv,iot_sensors_extended.csv,671,1793891445000
dbfs:/Volumes/ecomotive/landing/sensores/iot_sensors_incremental.csv,iot_sensors_incremental.csv,346,1793936841000
dbfs:/Volumes/ecomotive/landing/sensores/iot_sensors_prueba_incremental_30.csv,iot_sensors_prueba_incremental_30.csv,1746,1794103521000
dbfs:/Volumes/ecomotive/landing/sensores/iot_sensors_quality_issues.csv,iot_sensors_quality_issues.csv,517,1793891503000
dbfs:/Volumes/ecomotive/landing/sensores/iot_sensors_raw.csv,iot_sensors_raw.csv,979,1793885587000


Ubicación del esquema:
/Volumes/ecomotive/metadata/schemas/sensores_pipeline/
Ubicación del checkpoint:
/Volumes/ecomotive/metadata/checkpoints


# 2. Crear las instrucciones del autoloader

In [0]:
%python
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

In [0]:
%python
df_stream.printSchema()

root
 |-- sensor_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- temperature_c: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- status: string (nullable = true)
 |-- battery_level_pct: string (nullable = true)
 |-- firmware_version: string (nullable = true)
 |-- signal_strength_dbm: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



# 3. Configurar la escritura

In [0]:
%python
query=(
    df_stream.writeStream
    .format("Delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeschema", "true")
     .trigger(availableNow=True)
    .toTable(target_table)

)

In [0]:
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sensor_id) AS sensores_unicos
FROM ecomotive.bronze.iot_sensors_raw;

total_registros,sensores_unicos
103,37


In [0]:
SELECT *
FROM ecomotive.bronze.iot_sensors_raw


sensor_id,truck_id,timestamp,temperature_c,latitude,longitude,status,battery_level_pct,firmware_version,signal_strength_dbm,_rescued_data
S-1001,T-101,2025-01-12T08:00:00,2.5,40.4168,-3.7038,OK,null,null,null,null
S-1001,T-101,2025-01-12T08:15:00,2.8,40.4200,-3.7100,OK,null,null,null,null
S-1001,T-101,2025-01-12T08:30:00,3.1,40.4300,-3.7200,OK,null,null,null,null
S-1002,T-102,2025-01-12T09:00:00,3.9,41.3851,2.1734,WARNING,null,null,null,null
S-1002,T-102,2025-01-12T09:15:00,4.2,41.3900,2.1800,ALARM,null,null,null,null
S-1002,T-102,2025-01-12T09:30:00,4.5,41.3950,2.1850,ALARM,null,null,null,null
S-1003,T-103,2025-01-12T08:10:00,-1.0,39.4699,-0.3763,OK,null,null,null,null
S-1004,T-104,2025-01-12T10:00:00,2.0,37.3891,-5.9845,OK,null,null,null,null
S-1005,T-105,2025-01-12T11:00:00,2.2,43.2630,-2.9350,OK,null,null,null,null
S-1001,T-101,2025-01-12T08:45:00,3.0,40.4400,-3.7300,OK,null,null,null,null


In [0]:
SELECT
    COUNT(*) AS total,
    COUNT(battery_level_pct) AS registros_con_bateria,
    COUNT(firmware_version) AS registros_con_firmware,
    COUNT(signal_strength_dbm) AS registros_con_senal
FROM ecomotive.bronze.iot_sensors_raw;

total,registros_con_bateria,registros_con_firmware,registros_con_senal
103,21,21,21


In [0]:
SELECT
    COUNT(*) AS total,
    COUNT(sensor_id) AS sensores_con_id,
    COUNT(temperature_c) AS temperaturas_no_nulas,
    COUNT(timestamp) AS fechas_no_nulas
FROM ecomotive.bronze.iot_sensors_raw;

total,sensores_con_id,temperaturas_no_nulas,fechas_no_nulas
103,96,102,103


In [0]:
%python
display(dbutils.fs.ls(checkpoint_path))

path,name,size,modificationTime
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/,commits/,0,1794190472689
dbfs:/Volumes/ecomotive/metadata/checkpoints/metadata,metadata,45,1793894394000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/,offsets/,0,1794190472689
dbfs:/Volumes/ecomotive/metadata/checkpoints/sensores_pipeline/,sensores_pipeline/,0,1794190472689
dbfs:/Volumes/ecomotive/metadata/checkpoints/sources/,sources/,0,1794190472689


# 4. Validaciones a través del checkPoint

In [0]:
%python
# 4. Validaciones a través del checkPoint

display(dbutils.fs.ls(checkpoint_path + "/offsets/"))

path,name,size,modificationTime
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/0,0,2735,1793894399000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/1,1,2736,1793936952000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/2,2,2736,1794097958000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/3,3,2736,1794103566000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/4,4,2736,1794190319000


In [0]:
%python
display(dbutils.fs.ls(checkpoint_path + "/commits/"))

path,name,size,modificationTime
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/0,0,29,1793894404000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/1,1,29,1793936957000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/2,2,29,1794097963000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/3,3,29,1794103569000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/4,4,29,1794190322000


In [0]:
%python
display(dbutils.fs.ls(checkpoint_path + "/sources/"))

path,name,size,modificationTime
dbfs:/Volumes/ecomotive/metadata/checkpoints/sources/0/,0/,0,1794190474993
